# Qwen3.6-27B Cookbook (API model: qwen3.6-27b)

This notebook is a practical starting point for building with Qwen3.6-27B via Alibaba Cloud Model Studio's OpenAI-compatible API.

What is covered:
- setup and auth
- basic chat completions
- streaming
- framework integration (LangChain)
- tool-enabled agent loop
- long-context QA pattern
- multimodal message shape
- token usage tracking

Source: https://qwen.ai/blog?id=qwen3.6-27b

## 0) Setup and Installation

In [ ]:
# Run this once in a fresh environment.
# %pip install -U openai langchain langchain-openai tiktoken

In [ ]:
import os
from openai import OpenAI

MODEL = "qwen3.6-27b"
BASE_URL = os.getenv("DASHSCOPE_BASE_URL", "https://dashscope-intl.aliyuncs.com/compatible-mode/v1")
API_KEY = os.getenv("DASHSCOPE_API_KEY")

if not API_KEY:
    raise ValueError("Set DASHSCOPE_API_KEY before running this notebook.")

client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
print(f"Using model: {MODEL}")
print(f"Base URL: {BASE_URL}")

## 1) Basic Usage with Provider SDK

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise coding assistant."},
        {"role": "user", "content": "List practical steps to add an AI code review gate in CI."},
    ],
    extra_body={"enable_thinking": True},
)
print(response.choices[0].message.content)

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Give me 5 practical checks before shipping a coding agent to production."}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="")
print()

### Preserve thinking across turns

Qwen3.6-27B docs mention `preserve_thinking` for multi-turn agentic tasks.

In [ ]:
messages = [
    {"role": "system", "content": "You are a senior backend engineer."},
    {"role": "user", "content": "Design an idempotent webhook retry policy."},
]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    extra_body={"enable_thinking": True, "preserve_thinking": True},
)

messages.append({"role": "assistant", "content": first.choices[0].message.content})
messages.append({"role": "user", "content": "Now optimize this for high-throughput event ingestion."})

second = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    extra_body={"enable_thinking": True, "preserve_thinking": True},
)
print(second.choices[0].message.content)

## 2) Framework Integration

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0,
)

msg = llm.invoke("Write a migration checklist for upgrading to a stronger coding model in CI/CD.")
print(msg.content)

## 3) Building Agents

In [ ]:
import json

def get_package_health(package_name: str):
    fake_registry = {"fastapi": "healthy", "numpy": "healthy", "legacy-sdk": "deprecated"}
    return {"package": package_name, "status": fake_registry.get(package_name, "unknown")}

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_package_health",
            "description": "Return support status for a package name.",
            "parameters": {
                "type": "object",
                "properties": {"package_name": {"type": "string"}},
                "required": ["package_name"],
            },
        },
    }
]

In [ ]:
messages = [
    {"role": "system", "content": "You are a release assistant that verifies dependencies before deployment."},
    {"role": "user", "content": "Can I ship a service that depends on legacy-sdk?"},
]

first = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

message = first.choices[0].message
messages.append(message.model_dump())

if message.tool_calls:
    for call in message.tool_calls:
        if call.function.name == "get_package_health":
            args = json.loads(call.function.arguments)
            result = get_package_health(args["package_name"])
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": json.dumps(result),
                }
            )

final = client.chat.completions.create(model=MODEL, messages=messages)
print(final.choices[0].message.content)

## 4) Advanced Applications

In [ ]:
def chunk_text(text: str, chunk_size: int = 2000):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

# Replace with your actual long document text.
long_doc = """
Qwen3.6-27B is a dense 27B multimodal model focused on strong agentic coding performance.
The release highlights OpenAI-compatible API usage and agent tool integrations.
""" * 300

chunks = chunk_text(long_doc)
context = "\n\n".join(chunks[:20])

qa = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer using only the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: What are the main practical takeaways for an engineering team?"},
    ],
)
print(qa.choices[0].message.content)

In [ ]:
# Multimodal input shape example (text + image).
# Replace IMAGE_URL with a real public image URL.
IMAGE_URL = "https://example.com/diagram.png"

mm = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "Summarize this architecture diagram and list two risks."},
                {"type": "image_url", "image_url": {"url": IMAGE_URL}},
            ],
        }
    ],
)
print(mm.choices[0].message.content)

## Model-Specific Notes

- Open-source checkpoint: Qwen3.6-27B (dense 27B parameters).
- API model string shown in docs: `qwen3.6-27b`.
- Native multimodal support for text, image, and video understanding.
- Release docs call out `enable_thinking` and `preserve_thinking` for agentic workflows.
- Build section highlights integration patterns for OpenClaw, Qwen Code, and Claude Code.
- OpenClaw sample config in the post shows `contextWindow: 131072`; verify endpoint limits before production rollout.

In [ ]:
usage_demo = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Return 3 bullet points on rollout risk management."}],
)

usage = usage_demo.usage
print("prompt_tokens:", usage.prompt_tokens)
print("completion_tokens:", usage.completion_tokens)
print("total_tokens:", usage.total_tokens)

---
Cookbook done. Next step: swap placeholders with your real task payloads and run each section end-to-end in your target environment.